In [ ]:
import esm
import torch

import reverse_distillation

# Load ESM-2 model
esm2_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
rd_model, alphabet = reverse_distillation.pretrained.esm2_rd_650M()

batch_converter = alphabet.get_batch_converter()
esm2_model.eval()  # disables dropout for deterministic results
rd_model.eval()  # disables dropout for deterministic results

/hpc/home/dgc26/projects/esm-scaling-v2/src/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Prepare data (first 2 sequences from ESMStructuralSplitDataset superfamily / 4)
data = [
    ("protein1", "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"),
    (
        "protein2",
        "KALTARQQEVFDLIRDHISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE",
    ),
    (
        "protein2 with mask",
        "KALTARQQEVFDLIRD<mask>ISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE",
    ),
    ("protein3", "K A <mask> I S Q"),
]
batch_labels, batch_strs, batch_tokens = batch_converter(data)
batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

In [ ]:
# Extract per-residue representations (on CPU)
with torch.no_grad():
    results_esm = esm2_model(batch_tokens, repr_layers=[33], return_contacts=True)
    results_rd = rd_model(batch_tokens)

esm_token_representations = results_esm["representations"][33]
rd_token_representations = results_rd["representations"]["650M"]

In [ ]:
# Generate per-sequence representations via averaging
# NOTE: token 0 is always a beginning-of-sequence token, so the first residue is token 1.
sequence_representations = []
for i, tokens_len in enumerate(batch_lens):
    print(
        f"esm representation size: {esm_token_representations[i, 1 : tokens_len - 1].size()}"
    )
    print(
        f"rd representation size: {rd_token_representations[i, 1 : tokens_len - 1].size()}"
    )